# jstark — GroceryFeatures demo

[jstark](https://github.com/jamiekt/jstark) is a PySpark library for generating time-based features for machine learning. Features are calculated relative to an **as at** date, enabling point-in-time feature engineering over configurable time windows.

This notebook walks through `GroceryFeatures` using the sample `FakeGroceryTransactions` data. For the full list of available features see the [README](https://github.com/jamiekt/jstark#features-reference).

In [ ]:
from datetime import date

from jstark.grocery import GroceryFeatures
from jstark.sample.transactions import FakeGroceryTransactions

## Sample data

`FakeGroceryTransactions` produces a deterministic PySpark DataFrame of synthetic grocery transactions (baskets, stores, products, spend) suitable for demonstration.

In [ ]:
input_df = FakeGroceryTransactions(seed=42, number_of_baskets=500).df
input_df.printSchema()
input_df.show(5)

## Feature period mnemonics

Feature names end with a mnemonic `{start}{unit}{end}` where the unit is one of `d` (days), `w` (weeks), `m` (months), `q` (quarters) or `y` (years).

Examples, all relative to the `as_at` date:

- `3m1` — 3 months before to 1 month before
- `6m4` — 6 months before to 4 months before
- `1q1` — the single full quarter preceding `as_at`

In [ ]:
gf = GroceryFeatures().with_as_at(date(2022, 1, 1)).with_feature_periods(["3m1", "6m4"])

In [ ]:
output_df = input_df.groupBy("Store").agg(*gf.features)
output_df.select(
    "Store",
    "BasketCount_3m1",
    "BasketCount_6m4",
    "GrossSpend_3m1",
    "GrossSpend_6m4",
).show()

## Feature descriptions

Every feature carries a human-readable description in its PySpark column metadata.

In [ ]:
for c in output_df.schema:
    if c.name.endswith("_3m1"):
        print(f"{c.name}: {c.metadata['description']}")

## Feature input references

`gf.references` tells you which input columns a feature depends on.

In [ ]:
print(gf.references["BasketCount_3m1"])
print(gf.references["CustomerCount_3m1"])
print(gf.references["AvgGrossSpendPerBasket_3m1"])

## Absolute period labels

By default, feature names carry the mnemonic of their period (e.g. `BasketCount_1q1`). Calling `with_use_absolute_periods(True)` rewrites the suffix as the concrete calendar period the mnemonic resolves to, given `as_at`. For example, with `as_at=2022-01-01` the mnemonic `1q1` becomes `2021Q4` and `2q2` becomes `2021Q3`.

In [ ]:
gf_abs = (
    GroceryFeatures()
    .with_as_at(date(2022, 1, 1))
    .with_feature_periods(["1q1", "2q2"])
    .with_use_absolute_periods(True)
)
output_abs_df = input_df.groupBy("Store").agg(*gf_abs.features)
output_abs_df.select("Store", "BasketCount_2021Q4", "BasketCount_2021Q3").show()

## Next steps

See the [Features reference](https://github.com/jamiekt/jstark#features-reference) in the project README for the full catalogue of grocery features.